## Install Packages

In [1]:
%pip install --upgrade --quiet  langchain langchain-community langchainhub langchain-openai langchain-chroma bs4

## Based on [Langchain Tutorial](https://python.langchain.com/docs/use_cases/question_answering/quickstart/)

## Load Packages

In [2]:
import bs4
from langchain import hub
from langchain_community.document_loaders import WebBaseLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Set OpenAI Key

Create a dev account that will give you credit and access to OpenAI models

In [3]:
import os
import dotenv

dotenv.load_dotenv("env")
#os.environ["OPENAI_API_KEY"] = ""

True

## Select LLM Model

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

## Load and Parse Data
https://lilianweng.github.io/posts/2023-06-23-agent/


In [5]:
# Collects data from site using bs4
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

In [6]:
print(docs[0].page_content[:500])



      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In


## Split data and load into Vector Database

In [7]:
# Text splitter method
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
                                               chunk_overlap=200)

## Vector Database

[Vector Space](https://openai.com/blog/introducing-text-and-code-embeddings#:~:text=Drag%20to%20pan%2C%20scroll%20or%20pinch%20to%20zoom)

In [8]:
splits = text_splitter.split_documents(docs)
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

In [9]:
# Retrieve to pull relevant snippets of the blog.
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})

## Prompting

In [10]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [11]:
from langchain_core.prompts import PromptTemplate

template = """Using only the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Keep the answer as detailed as possible.

Context: {context}

Question: {question}

Answer:"""

custom_rag_prompt = PromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | custom_rag_prompt
    | llm
    | StrOutputParser()
)

## Asking questions

In [12]:
retrieved_docs = retriever.invoke("What are common ANN algorithms?")
retrieved_docs

[Document(page_content='Maximum Inner Product Search (MIPS)#\nThe external memory can alleviate the restriction of finite attention span.  A standard practice is to save the embedding representation of information into a vector store database that can support fast maximum inner-product search (MIPS). To optimize the retrieval speed, the common choice is the approximate nearest neighbors (ANN)\u200b algorithm to return approximately top k nearest neighbors to trade off a little accuracy lost for a huge speedup.\nA couple common choices of ANN algorithms for fast MIPS:', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}),
 Document(page_content='LSH (Locality-Sensitive Hashing): It introduces a hashing function such that similar input items are mapped to the same buckets with high probability, where the number of buckets is much smaller than the number of inputs.\nANNOY (Approximate Nearest Neighbors Oh Yeah): The core data structure are random projection trees,

In [13]:
rag_chain.invoke("What are common ANN algorithms?")

'Common ANN algorithms for fast Maximum Inner Product Search (MIPS) include Locality-Sensitive Hashing (LSH), ANNOY (Approximate Nearest Neighbors Oh Yeah), FAISS (Facebook AI Similarity Search), and ScaNN (Scalable Nearest Neighbors). These algorithms are used to optimize retrieval speed by returning approximately top k nearest neighbors, trading off a little accuracy for a significant speedup. LSH introduces a hashing function to map similar input items to the same buckets with high probability. ANNOY uses random projection trees to search through the input space iteratively. FAISS applies vector quantization to partition the vector space into clusters. ScaNN utilizes anisotropic vector quantization to quantize data points efficiently. Each of these algorithms has its unique approach to solving the MIPS problem.'

In [14]:
rag_chain.invoke("What is FAISS?")

'FAISS (Facebook AI Similarity Search) is an algorithm that operates under the assumption that in high-dimensional space, distances between nodes follow a Gaussian distribution, allowing for the clustering of data points. It applies vector quantization by partitioning the vector space into clusters and then refining the quantization within those clusters. The search process involves looking for cluster candidates with coarse quantization first and then further examining each cluster with finer quantization. It aims to efficiently search for similar vectors in high-dimensional spaces by leveraging the clustering structure of the data points.'